In [1]:
# ==========================================
# PROYECTO FINAL IA GENERATIVA - CONFIGURACIÓN
# ==========================================

uuid_alumno = "917343c0-dc7f-4300-85cd-651670e304c2" 

import os
from dotenv import load_dotenv

# Cargar el archivo .env
load_dotenv()

# Leer la variable exacta que tienes en tu archivo
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("Error: No se encuentra 'GEMINI_API_KEY'. Verifica el nombre en el archivo .env")
else:
    # Asignarla a la variable que usará LangChain (GOOGLE_API_KEY)
    os.environ["GOOGLE_API_KEY"] = api_key
    print("¡ÉXITO! API Key cargada correctamente.")

¡ÉXITO! API Key cargada correctamente.


In [7]:
# ==========================================
# PASO 2 Y 3: CARGA Y CREACIÓN DE CHROMA DB
# ==========================================

import json
import os
import time
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

carpeta_datos = "data"
archivos_json = ['league_of_legends_champions.json', 'league_of_legends_items.json', 'league_of_legends_macro.json']
documentos_totales = []

for archivo in archivos_json:
    ruta_completa = os.path.join(carpeta_datos, archivo)
    try:
        with open(ruta_completa, 'r', encoding='utf-8') as f:
            datos = json.load(f)
            for item in datos:
                contenido = json.dumps(item, ensure_ascii=False)
                doc = Document(page_content=contenido, metadata={"source": archivo})
                documentos_totales.append(doc)
        print(f"✅ Cargado: {ruta_completa}")
    except FileNotFoundError:
        print(f"❌ Error: No se encuentra el archivo en {ruta_completa}")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documentos_totales)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

chunk_length = 5
for i in range(0, len(chunks), chunk_length):
    lote = chunks[i:i + chunk_length]
    vector_db.add_documents(lote)
    time.sleep(6)
    print(f"Procesado lote {i // chunk_length + 1}")

print(f"\n🚀 Base de conocimiento creada con {len(chunks)} fragmentos.")

test_query = "Cuál es la mejor build para un campeón?"
resultados = vector_db.similarity_search(test_query, k=2)

print(f"\n🔍 Test de recuperación para: '{test_query}'")
for i, res in enumerate(resultados):
    print(f"Resultado {i+1}: {res.page_content[:100]}...")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Cargado: data\league_of_legends_champions.json
✅ Cargado: data\league_of_legends_items.json
✅ Cargado: data\league_of_legends_macro.json
Procesado lote 1
Procesado lote 2
Procesado lote 3
Procesado lote 4
Procesado lote 5
Procesado lote 6
Procesado lote 7
Procesado lote 8
Procesado lote 9
Procesado lote 10
Procesado lote 11
Procesado lote 12
Procesado lote 13
Procesado lote 14
Procesado lote 15
Procesado lote 16
Procesado lote 17
Procesado lote 18
Procesado lote 19
Procesado lote 20
Procesado lote 21
Procesado lote 22
Procesado lote 23
Procesado lote 24
Procesado lote 25
Procesado lote 26
Procesado lote 27
Procesado lote 28
Procesado lote 29
Procesado lote 30
Procesado lote 31
Procesado lote 32
Procesado lote 33
Procesado lote 34
Procesado lote 35
Procesado lote 36
Procesado lote 37
Procesado lote 38
Procesado lote 39
Procesado lote 40
Procesado lote 41
Procesado lote 42
Procesado lote 43
Procesado lote 44
Procesado lote 45
Procesado lote 46
Procesado lote 47
Procesado lote 48
Proces

In [ ]:
import os
from typing import TypedDict
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

class AgentState(TypedDict):
    question: str
    context: str
    answer: str

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)

retriever = vector_db.as_retriever(search_kwargs={"k": 3})

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

system_prompt_rag = (
    "Eres un Analista Experto en League of Legends de Nivel Competitivo y un asistente RAG especializado. "
    "Tu único objetivo es responder a las consultas de los usuarios utilizando exclusivamente los datos proporcionados en el contexto adjunto.\n\n"
    "Pautas de comportamiento obligatorias:\n"
    "1. Confianza Absoluta en el Contexto: Si los datos necesarios para responder a la pregunta no se encuentran explícitamente en el fragmento de texto proporcionado, tu respuesta debe ser textualmente: 'No dispongo de esa información en mi base de conocimiento local.' No intentes rellenar huecos, deducir datos no escritos ni inventar mecánicas.\n"
    "2. Neutralidad y Rigor: Mantén un tono técnico, analítico y directo. Evita opiniones personales que no estén respaldadas por los JSON de origen.\n"
    "3. Tratamiento de Campeones y Objetos: Si te preguntan por un campeón u objeto que aparece en el contexto pero los detalles específicos (como sus builds o estadísticas completas) faltan, indícalo detallando qué información exacta sí tienes disponible.\n\n"
    "Formato de Respuesta:\n"
    "- Usa negritas para destacar palabras clave, nombres de campeones, habilidades u objetos.\n"
    "- Si vas a listar elementos o estadísticas, estructúralos siempre en listas de puntos limpios.\n"
    "- Evita párrafos excesivamente largos; prioriza la legibilidad inmediata.\n\n"
    "Contexto adjunto:\n"
    "{context}"
)

system_prompt_fallback = (
    "Eres un Analista Experto en League of Legends de Nivel Competitivo.\n\n"
    "La base de conocimiento local no contiene datos específicos para esta consulta. Responde utilizando tu conocimiento general del meta de forma extremadamente concisa, directa y al grano.\n\n"
    "Pauta obligatoria: Comienza tu respuesta integrando de forma fluida y profesional en el primer párrafo que, ante la falta de registros específicos en la base local, vas a ofrecer un análisis estratégico basado en los fundamentos generales del juego competitivo. No uses corchetes ni etiquetas robóticas.\n\n"
    "Restricciones de formato:\n"
    "- Limita tu recomendación a un máximo de 2 campeones.\n"
    "- Para cada campeón, aporta únicamente dos viñetas breves con la razón táctica básica.\n"
    "- No te extiendas con introducciones extensas ni conclusiones innecesarias."
)

prompt_rag = ChatPromptTemplate.from_messages([
    ("system", system_prompt_rag),
    ("human", "{input}"),
])

prompt_fallback = ChatPromptTemplate.from_messages([
    ("system", system_prompt_fallback),
    ("human", "{input}"),
])

def retrieve_node(state: AgentState):
    question = state["question"]
    try:
        docs = retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in docs)
    except Exception:
        context = ""
    return {"context": context}

def generate_rag_node(state: AgentState):
    question = state["question"]
    context = state["context"]
    chain = prompt_rag | llm
    response = chain.invoke({"context": context, "input": question})
    return {"answer": response.content}

def generate_fallback_node(state: AgentState):
    question = state["question"]
    chain = prompt_fallback | llm
    response = chain.invoke({"input": question})
    return {"answer": response.content}

def checker_router(state: AgentState):
    answer_str = str(state["answer"]).lower()
    if "no dispongo de esa información" in answer_str:
        return "fallback"
    return "end"

workflow = StateGraph(AgentState)

workflow.add_node("retrieve", retrieve_node)
workflow.add_node("generate_rag", generate_rag_node)
workflow.add_node("generate_fallback", generate_fallback_node)

workflow.add_edge(START, "retrieve")
workflow.add_edge("retrieve", "generate_rag")

workflow.add_conditional_edges(
    "generate_rag",
    checker_router,
    {
        "fallback": "generate_fallback",
        "end": END
    }
)

workflow.add_edge("generate_fallback", END)

app = workflow.compile()

inputs = {"question": "Todo mi equipo es AP, que campeón debería jugar yo estando en la top lane?"}
resultado = app.invoke(inputs)

print(resultado["answer"])

[{'type': 'text', 'text': "[Nota del Analista: Información basada en conocimiento general de juego, no en la base de datos local]\n\nTener un equipo completamente AP (Poder de Habilidad) es una de las mayores desventajas estratégicas en League of Legends. Permite que el equipo enemigo optimice su build comprando resistencia mágica barata y altamente eficiente (como *Kaenic Rookern* o *Mercury's Treads*), lo que reducirá drásticamente el daño de todo tu equipo. \n\nComo carrilero superior, tienes la responsabilidad de ser la **amenaza de daño físico (AD)** que obligue al enemigo a diversificar sus objetos de defensa (comprando armadura) o a sufrir las consecuencias en el juego medio y tardío.\n\nA continuación, te presento las mejores opciones según tu estilo de juego, analizadas bajo tres pilares fundamentales: **daño AD**, **sinergia** y **fase de líneas**.\n\n---\n\n### Opción 1: Aatrox (El Coloso de Daño Físico y Sustento)\nEs actualmente una de las opciones de blind pick más segura